In [2]:
# run for only first time

# !git clone https://github.com/thuanz123/realfill.git

In [2]:
# run for only first time
# %cd realfill
# %ls

In [3]:
# !pip install -r /content/realfill/requirements.txt
!pip install -r requirements.txt

In [4]:
from accelerate.utils import write_basic_config
write_basic_config()

/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration already exists at /userhome/cs/zc923404/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


False

In [5]:
%pip install "numpy<2"
import numpy
numpy.__version__

Note: you may need to restart the kernel to use updated packages.


'1.26.4'

In [8]:
%pip install diffusers transformers accelerate scipy safetensors 

Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
import os
from diffusers import StableDiffusionInpaintPipeline

# 1. Define the official Model ID
model_id = "sd2-community/stable-diffusion-2-inpainting"

# 2. Check for device (NVIDIA CUDA for the farm, MPS for your Mac)
if torch.cuda.is_available():
    current_device = "cuda"
    current_dtype = torch.float16 # Farm GPUs handle float16 for speed
elif torch.backends.mps.is_available():
    current_device = "mps"
    current_dtype = torch.float32 # Mac is more stable with float32
else:
    current_device = "cpu"
    current_dtype = torch.float32

print(f"Loading pipeline onto: {current_device}")

# 3. Load the pipeline (It will download once and then cache automatically)
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=current_dtype,
    use_safetensors=True
)

pipe = pipe.to(current_device)

Loading pipeline onto: cuda


/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading pipeline components...: 100%|██████████████████████████████████| 6/6 [00:01<00:00,  4.67it/s]


In [ ]:
# from huggingface_hub import login

# base brain: stable diffusion 2.0 inpainting
%env MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
# reference image
%env TRAIN_DIR=data/flowerwoman
%env OUTPUT_DIR=flowerwoman-model

# %env TRAIN_DIR=data/chiwah
# %env OUTPUT_DIR=chiwah-model

# !accelerate launch train_realfill.py \
!accelerate launch train_realfill.py \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --train_data_dir=$TRAIN_DIR \
  --output_dir=$OUTPUT_DIR \
  --mixed_precision=fp16 \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --unet_learning_rate=2e-4 \
  --text_encoder_learning_rate=4e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=100 \
  --max_train_steps=2000 \
  --lora_rank=8 \
  --lora_dropout=0.1 \
  --lora_alpha=16

  #   # --train_batch_size=16 \

env: MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
env: TRAIN_DIR=data/flowerwoman
env: OUTPUT_DIR=flowerwoman-model
/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transfo

In [10]:
%env MODEL_PATH=chiwah-model
%env VAL_IMG=data/chiwah/target/target.png
%env VAL_MASK=data/chiwah/target/mask.png
%env OUTPUT_IMG_DIR=chiwah-results


!accelerate launch infer.py \
    --model_path=$MODEL_PATH \
    --validation_image=$VAL_IMG \
    --validation_mask=$VAL_MASK \
    --output_dir=$OUTPUT_IMG_DIR

env: MODEL_PATH=chiwah-model
env: VAL_IMG=data/chiwah/target/target.png
env: VAL_MASK=data/chiwah/target/mask.png
env: OUTPUT_IMG_DIR=chiwah-results
Using device: cuda
Traceback (most recent call last):
  File "/userhome/cs/zc923404/realfill/infer.py", line 67, in <module>
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
  File "/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/diffusers/pipelines/pipeline_utils.py", line 932, in from_pretrained
    cached_folder = cls.download(
  File "/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/diffusers/pipelines/pipeline_utils.py", line 1507, in download
    cached_folder = snapshot_download(
  File "/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validate_repo_id(arg_value)
  File "/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/huggingface_hub/utils/_validator

In [ ]:
!zip -r realfill.zip \chiwah-results
